In [ ]:
from agent import Agent
from yinshEnv import YinshEnv
from game import Game

class Evaluation:
    def __init__(self, model_1_path: str, model_2_path: str):
        self.model_1_path = model_1_path
        self.model_2_path = model_2_path

    def evaluate(self, n: int):
        """
        두 모델이 서로 번갈아가며 플레이하며 n회 평가 (각 모델이 흑/백 각각 수행)
        """
        score = {
            "model_1": 0,
            "model_2": 0,
            "draws": 0
        }

        agent_1 = Agent(local_predictions=True, model_path=self.model_1_path)
        agent_2 = Agent(local_predictions=True, model_path=self.model_2_path)

        for i in range(n):
            print(f"\n{'='*10} Match {i+1}/{n} {'='*10}")

            # Agent1 (White), Agent2 (Black)
            env = YinshEnv()
            game = Game(env, agent_1, agent_2)
            result = game.play_one_game(stochastic=False)
            if result == 1:
                score["model_1"] += 1
            elif result == -1:
                score["model_2"] += 1
            else:
                score["draws"] += 1

            # Agent2 (White), Agent1 (Black)
            env = YinshEnv()
            game = Game(env, agent_2, agent_1)
            result = game.play_one_game(stochastic=False)
            if result == 1:
                score["model_2"] += 1
            elif result == -1:
                score["model_1"] += 1
            else:
                score["draws"] += 1

        total_games = n * 2
        return (
            f"\n📊 Evaluation Summary ({total_games} games):\n"
            f"Model 1 ({self.model_1_path}): {score['model_1']} wins\n"
            f"Model 2 ({self.model_2_path}): {score['model_2']} wins\n"
            f"Draws: {score['draws']}\n"
        )

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Evaluate two Yinsh models")
    parser.add_argument("model_1", type=str, help="Path to model 1")
    parser.add_argument("model_2", type=str, help="Path to model 2")
    parser.add_argument("nr_games", type=int, help="Number of matchups (x2: each model plays white/black)")

    args = parser.parse_args()

    evaluator = Evaluation(args.model_1, args.model_2)
    result_summary = evaluator.evaluate(args.nr_games)
    print(result_summary)
